<a href="https://colab.research.google.com/github/ashishsantikari/lab-sql-query-from-table-names-continued/blob/main/lab-sql-query-from-table-names-continued.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL query from table names - Continued

In [8]:
from openai import OpenAI
import os
# from dotenv import load_dotenv, find_dotenv
# _ = load_dotenv(find_dotenv())
# OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

from google.colab import userdata
OPENAI_API_KEY  = userdata.get('OPENAI_API_KEY')

## The old Prompt

In [19]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [13]:
context = [ {'role':'system', 'content':"""
 CREATE TABLE `employees` (`ID_usr` INT, `name` VARCHAR);
 CREATE TABLE `salary` (`ID_usr` INT, `year` DATE, `salary` FLOAT);
 CREATE TABLE  `studies` (`ID` INT, `ID_usr` INT, `educational_level` INT, `Institution` VARCHAR, `Years` DATE, `Speciality` VARCHAR);
"""} ]



In [14]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
What is the median salary of an employee?
"""
})

In [15]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [17]:
#new
context_user = context.copy()
print(return_CCRMSQL("""What is the median salary of an employee?""", context_user))

```sql
WITH ranked_salaries AS (
    SELECT ID_usr, salary, ROW_NUMBER() OVER (PARTITION BY ID_usr ORDER BY salary) AS rn,
           COUNT(*) OVER (PARTITION BY ID_usr) AS cnt
    FROM salary
)
SELECT AVG(salary) AS median_salary
FROM (
    SELECT ID_usr, salary
    FROM ranked_salaries
    WHERE rn IN (FLOOR((cnt + 1) / 2), CEIL((cnt + 1) / 2))
) AS median_salaries;
```


In [21]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("What is the median salary of an employee?", old_context_user))

This is your SQL:
```sql
SELECT salary
FROM (
    SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS row_num,
    COUNT(*) OVER () AS total_rows
    FROM salary
) AS subquery
WHERE row_num IN (total_rows/2, (total_rows+1)/2);
```

This SQL query calculates the median salary of employees by ordering the salaries in the `salary` table and then selecting the middle value or the average of the two middle values if there is an even number of rows.


In [22]:
#new
print(return_CCRMSQL("What is the median salary of an employee?", context_user))

```sql
WITH RankedSalaries AS (
    SELECT ID_usr, salary, ROW_NUMBER() OVER (PARTITION BY ID_usr ORDER BY salary) AS rn,
           COUNT(*) OVER (PARTITION BY ID_usr) AS cnt
    FROM salary
)
SELECT AVG(salary) AS median_salary
FROM (
    SELECT ID_usr, salary
    FROM RankedSalaries
    WHERE cnt % 2 = 1 AND rn = (cnt + 1) / 2
    UNION ALL
    SELECT ID_usr, salary
    FROM RankedSalaries
    WHERE cnt % 2 = 0 AND rn IN (cnt / 2, cnt / 2 + 1)
) AS medians;
```


In [23]:
#old
print(return_CCRMSQL("What is the median salary of an employee?", old_context_user))

This is your SQL:
```sql
SELECT salary
FROM (
    SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS row_num,
    COUNT(*) OVER () AS total_rows
    FROM salary
) AS subquery
WHERE row_num IN (total_rows/2, (total_rows+1)/2);
```

This SQL query calculates the median salary of employees by ordering the salaries in the `salary` table and then selecting the middle value or the average of the two middle values if there is an even number of rows.


In [48]:
def line_sep(txt):
  str0 = ' '.join(("#"*38, txt.upper(), "#"*37))
  print("#"*80)
  print(str0)
  print("#"*80)

def compare_old_new(question):
  line_sep("new")
  context_user = context.copy()
  print(return_CCRMSQL(question, context_user))
  line_sep("old")
  old_context_user = old_context.copy()
  print(return_CCRMSQL(question, old_context_user))

In [49]:
compare_old_new("How is the distribution of education level of the employees?")

################################################################################
###################################### NEW #####################################
################################################################################
To determine the distribution of education levels of the employees, you can use the following SQL query:

```sql
SELECT educational_level, COUNT(*) AS count
FROM studies
GROUP BY educational_level;
```

This query will provide you with the count of employees for each educational level present in the `studies` table.
################################################################################
###################################### OLD #####################################
################################################################################
This is your SQL:
```sql
SELECT educational_level, COUNT(*) AS num_employees
FROM studies
GROUP BY educational_level;
```

Explanation: This SQL query selects the educational level of employees fr

In [50]:
compare_old_new("Which employees are under the age 25 ?")

################################################################################
###################################### NEW #####################################
################################################################################
To determine which employees are under the age of 25, we need to have a column in the `employees` table that stores the birth date of each employee. Without this information, we cannot accurately determine the age of the employees. 

If you provide the birth date information in the `employees` table, I can help you write a query to identify employees who are under the age of 25.
################################################################################
###################################### OLD #####################################
################################################################################
This is your SQL:
```sql
SELECT * 
FROM employees
WHERE age < 25;
```

Explanation: The SQL query selects all employees from the "em

In [52]:
compare_old_new("Where does the teachers live? What is the average distance of their home from school?")

################################################################################
###################################### NEW #####################################
################################################################################
I'm sorry, but the provided tables do not contain information about where the employees live or the distance of their homes from school. The tables only include data about employees' names, salaries, educational background, and institutions they attended. If you have any other questions related to the data in these tables, feel free to ask.
################################################################################
###################################### OLD #####################################
################################################################################
This is your SQL:
```sql
SELECT address, AVG(distance) AS average_distance
FROM teachers
GROUP BY address;
```

This SQL query selects the address of teachers and calculat

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?